# Monthly Seasonal Planning

This notebook performs comprehensive data quality checks on the Monthly_Seasonal_Planning dataset.


## 1. Data Loading


In [1]:
import pandas as pd

# Load the Monthly_Seasonal_Planning data
df = pd.read_csv('../datasets/Monthly_Seasonal_Planning.csv')

print("Data loaded successfully!")
print(f"Total records: {len(df)}")


Data loaded successfully!
Total records: 120


## 2. Basic Data Exploration


In [2]:
# Display first few rows
df.head()


,Month,Site ID,Product Category,Forecasted Sales,Actual Sales,Seasonal Adjustments
0,May-2024,PS3Y,Dairy,81260.50,96315.510888,0.1853
1,Dec-2024,W1Q5,Apparel,60609.93,64357.950697,0.0618
2,Sep-2024,T4UF,Apparel,84277.99,94950.573265,0.1266
3,Dec-2024,T4UF,Apparel,79067.48,82228.634271,0.0400
4,Mar-2025,OTOH,Apparel,50727.92,42559.446989,-0.1610


In [3]:
# Display column names
df.columns


Index(['Month', 'Site ID', 'Product Category', 'Forecasted Sales',
       'Actual Sales', 'Seasonal Adjustments'],
      dtype='object')

In [4]:
# Display dataset shape
df.shape


(120, 6)

In [5]:
# Display data types and basic info
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Month                 120 non-null    object 
 1   Site ID               120 non-null    object 
 2   Product Category      120 non-null    object 
 3   Forecasted Sales      120 non-null    float64
 4   Actual Sales          120 non-null    float64
 5   Seasonal Adjustments  120 non-null    float64
dtypes: float64(3), object(3)
memory usage: 5.8+ KB


## 3. Data Quality Checks

### 3.1 Missing Values Analysis


In [6]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

print("=== Missing Values Analysis ===")
print(missing_df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")


=== Missing Values Analysis ===
                      Missing Values  Percentage
Month                              0         0.0
Site ID                            0         0.0
Product Category                   0         0.0
Forecasted Sales                   0         0.0
Actual Sales                       0         0.0
Seasonal Adjustments               0         0.0

Total missing values: 0


### 3.2 Duplicate Records Check


In [7]:
# Check for duplicate records
duplicates_all = df.duplicated().sum()
print("=== Duplicate Records Check ===")
print(f"Total duplicate rows: {duplicates_all}")


=== Duplicate Records Check ===
Total duplicate rows: 0


### 3.3 Data Type Validation


In [8]:
# Check data types
print("=== Data Type Validation ===")
print(df.dtypes)

# Verify expected data types
expected_types = {
    'Month': 'object',
    'Site ID': 'object',
    'Product Category': 'object',
    'Forecasted Sales': 'float64',
    'Actual Sales': 'float64',
    'Seasonal Adjustments': 'float64'
}

type_issues = []
for col, expected in expected_types.items():
    if col in df.columns and df[col].dtype != expected:
        type_issues.append(f"{col}: expected {expected}, got {df[col].dtype}")

if type_issues:
    print("\nData Type Issues:")
    for issue in type_issues:
        print(f"  - {issue}")
else:
    print("\nAll data types are correct!")


=== Data Type Validation ===
Month                    object
Site ID                  object
Product Category         object
Forecasted Sales        float64
Actual Sales            float64
Seasonal Adjustments    float64
dtype: object

All data types are correct!


### 3.4 Value Range Checks


In [9]:
# Check value ranges for numeric columns
print("=== Value Range Checks ===")

# Get numeric columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    print(f"\n{col}: Min={min_val}, Max={max_val}")
    
    # Check for negative values (except where appropriate)
    if 'ID' not in col and 'Flag' not in col:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  Negative values found: {neg_count}")


=== Value Range Checks ===

Forecasted Sales: Min=10647.46, Max=97540.67

Actual Sales: Min=9809.20846892493, Max=106745.56016228149

Seasonal Adjustments: Min=-0.1995, Max=0.1987
  Negative values found: 63


### 3.5 Categorical Value Validation


In [10]:
# Check unique values in categorical columns
print("=== Categorical Value Validation ===")

cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if 'ID' not in col and 'Date' not in col and 'Name' not in col:
        unique_vals = df[col].unique()
        print(f"\n{col} unique values: {unique_vals}")


=== Categorical Value Validation ===

Month unique values: ['May-2024' 'Dec-2024' 'Sep-2024' 'Mar-2025' 'Apr-2024' 'Jul-2024'
 'Feb-2025' 'Nov-2024' 'Aug-2024' 'Jun-2024' 'Jan-2025' 'Oct-2024']

Product Category unique values: ['Dairy' 'Apparel' 'Bakery' 'Electronics']


### 3.6 Outlier Detection


In [11]:
# Detect outliers using IQR method
print("=== Outlier Detection (IQR Method) ===")

def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

# Check outliers for numeric columns
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_columns:
    if 'ID' not in col:
        count, lower, upper = detect_outliers_iqr(col)
        print(f"\n{col}:")
        print(f"  Outliers: {count}")
        print(f"  Lower Bound: {lower:.2f}")
        print(f"  Upper Bound: {upper:.2f}")


=== Outlier Detection (IQR Method) ===

Forecasted Sales:
  Outliers: 0
  Lower Bound: -30737.18
  Upper Bound: 139415.69

Actual Sales:
  Outliers: 0
  Lower Bound: -21322.76
  Upper Bound: 131163.92

Seasonal Adjustments:
  Outliers: 0
  Lower Bound: -0.39
  Upper Bound: 0.37


## 4. Data Quality Summary


In [12]:
# Generate comprehensive data quality summary
print("=" * 50)
print("DATA QUALITY SUMMARY")
print("=" * 50)

print(f"\n1. Dataset Overview:")
print(f"   - Total Records: {len(df)}")
print(f"   - Total Columns: {len(df.columns)}")

print(f"\n2. Missing Values:")
print(f"   - Total Missing: {df.isnull().sum().sum()}")
for col in df.columns:
    missing = df[col].isnull().sum()
    print(f"   - {col}: {missing}")

print(f"\n3. Duplicate Records:")
print(f"   - Duplicate Rows: {df.duplicated().sum()}")

print(f"\n4. Data Quality Status:")
quality_passed = True

if df.isnull().sum().sum() > 0:
    print(f"   ❌ Missing values found")
    quality_passed = False
else:
    print(f"   ✓ No missing values")

if df.duplicated().sum() > 0:
    print(f"   ❌ Duplicate records found")
    quality_passed = False
else:
    print(f"   ✓ No duplicate records")

if quality_passed:
    print(f"\n✅ DATA QUALITY CHECK PASSED!")
else:
    print(f"\n❌ DATA QUALITY ISSUES DETECTED - REVIEW REQUIRED!")


DATA QUALITY SUMMARY

1. Dataset Overview:
   - Total Records: 120
   - Total Columns: 6

2. Missing Values:
   - Total Missing: 0
   - Month: 0
   - Site ID: 0
   - Product Category: 0
   - Forecasted Sales: 0
   - Actual Sales: 0
   - Seasonal Adjustments: 0

3. Duplicate Records:
   - Duplicate Rows: 0

4. Data Quality Status:
   ✓ No missing values
   ✓ No duplicate records

✅ DATA QUALITY CHECK PASSED!
